In [ ]:
!pip install -q qiskit qiskit-machine-learning kagglehub torchvision
!pip -q install torchlibrosa soundfile torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 7.3 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

import soundfile as sf
import torchaudio.transforms as T

from kagglehub import dataset_download
from sklearn.metrics import classification_report, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit import QuantumCircuit

In [ ]:
BATCH_SIZE = 8
FEATURE_DIM = 8          # ⚠️ keep ≤ 8
EPOCHS = 1
LR = 1e-4

IMAGE_SIZE = 224
TARGET_SR = 4000

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CNN device:", device)

CNN device: cuda


In [ ]:
dataset_root = dataset_download("arashnic/lung-dataset")
data_path = os.path.join(dataset_root, "Audio Files")
print("Total files:", len(os.listdir(data_path)))

CANONICAL_MAP = {
    "asthma": "Asthma",
    "copd": "COPD",
    "bron": "Bronchitis",
    "heart failure": "Heart Failure",
    "lung fibrosis": "Lung Fibrosis",
    "pleural effusion": "Pleural Effusion",
    "pneumonia": "Pneumonia",
    "n": "Normal"
}

Using Colab cache for faster access to the 'lung-dataset' dataset.
Total files: 336


In [ ]:
from collections import defaultdict
import random

class LungAudioDataset(Dataset):
    def __init__(self, root, max_per_class=200, seed=42):
        self.samples = []
        self.labels = []
        self.class_counts = defaultdict(int)

        random.seed(seed)

        all_files = [
            f for f in os.listdir(root)
            if f.lower().endswith(".wav")
        ]

        parsed = []

        for fname in all_files:
            lower = fname.lower()

            matched_label = None
            for key in CANONICAL_MAP:
                if key in lower:
                    matched_label = CANONICAL_MAP[key]
                    break

            if matched_label is None:
                continue  # skip unknown files

            parsed.append((os.path.join(root, fname), matched_label))

        # -------------------------
        # Build dynamic class list
        # -------------------------
        self.class_names = sorted(list(set(label for _, label in parsed)))
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}

        # -------------------------
        # Balanced sampling
        # -------------------------
        per_class_files = defaultdict(list)
        for path, label in parsed:
            per_class_files[label].append(path)

        for label in self.class_names:
            files = per_class_files[label]

            if len(files) < max_per_class:
                selected = files
            else:
                selected = random.sample(files, max_per_class)

            for f in selected:
                self.samples.append(f)
                self.labels.append(self.class_to_idx[label])
                self.class_counts[label] += 1

        print("Detected classes:", self.class_names)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wav = load_wav_fixed(self.samples[idx])
        y = self.labels[idx]
        return wav, y

In [ ]:
full_dataset = LungAudioDataset(data_path, max_per_class=200)

TARGET_CLASSES = full_dataset.class_names
NUM_CLASSES = len(TARGET_CLASSES)

print("\nDisease Classes:")
for i, c in enumerate(TARGET_CLASSES):
    print(f"  {i}: {c}")

print("\nFiles per class:")
for class_name in TARGET_CLASSES:
    print(f"  {class_name}: {full_dataset.class_counts[class_name]}")

In [ ]:
# def parse_disease_label(filename):
#     name = filename.replace(".wav", "")
#     _, rest = name.split("_", 1)

#     raw = rest.split(",")[0].lower()
#     raw = raw.replace("+", " and ").replace("&", " and ")

#     if "heart failure" in raw and "copd" in raw:
#         return "Heart Failure + COPD"

#     if "heart failure" in raw and "lung fibrosis" in raw:
#         return "Heart Failure + Lung Fibrosis"

#     if "asthma" in raw and "lung fibrosis" in raw:
#         return "Asthma and Lung Fibrosis"

#     for key, canonical in CANONICAL_MAP.items():
#         if key in raw:
#             return canonical

#     raise ValueError(f"Unknown disease label in file: {filename}")

In [ ]:
# class LungAudioDataset(Dataset):
#     def __init__(self, root):
#         self.samples = []
#         self.labels = []

#         diseases = []

#         for f in os.listdir(root):
#             if f.lower().endswith(".wav"):
#                 disease = parse_disease_label(f)
#                 self.samples.append(os.path.join(root, f))
#                 self.labels.append(disease)
#                 diseases.append(disease)

#         self.class_names = sorted(list(set(diseases)))
#         self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}
#         self.labels = [self.class_to_idx[l] for l in self.labels]

#         print("Detected classes:")
#         for i, c in enumerate(self.class_names):
#             print(f"  {i}: {c}")

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         wav = load_wav_fixed(self.samples[idx])  # [T]
#         label = self.labels[idx]
#         return wav, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

full_dataset = LungAudioDataset(data_path)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(full_dataset.class_names)

Detected classes:
  0: Asthma
  1: Asthma and Lung Fibrosis
  2: Bronchitis
  3: COPD
  4: Heart Failure
  5: Heart Failure + COPD
  6: Heart Failure + Lung Fibrosis
  7: Lung Fibrosis
  8: Normal
  9: Pneumonia


In [ ]:
feature_map = ZZFeatureMap(FEATURE_DIM, reps=1)
ansatz = RealAmplitudes(FEATURE_DIM, reps=1)

qc = QuantumCircuit(FEATURE_DIM)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

/tmp/ipython-input-1793363023.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(FEATURE_DIM, reps=1)
/tmp/ipython-input-1793363023.py:2: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(FEATURE_DIM, reps=1)


In [ ]:
qc.decompose().draw()

┌───┐┌───────────┐                                             »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_6: ┤ H ├┤ P(2*x[6]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_7: ┤ H ├┤ P(2*x[7]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«                                                  »
«q_0: ────────────────────────────────■─────────■──»
«                                     │         │  »
«q_1: ────────────────────────────────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐┌─┴─┐  │  »
«q_2: ┤ P(2*(π - x[0])*(π - x[2])) ├┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘└───┘└───┘┌─┴─┐»
«q_3: ────────────────────────────────────────┤ X ├»
«                                             └───┘»
«q_4: ─────────────────────────────────────────────»
«                                                  »
«q_5: ─────────────────────────────────────────────»
«                                                  »
«q_6: ─────────────────────────────────────────────»
«                                                  »
«q_7: ─────────────────────────────────────────────»
«                                                  »
«                                                       »
«q_0: ─────────────────────────────────────■─────────■──»
«                                          │         │  »
«q_1: ────────────────────────────────■────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │    │    │  »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──┼────┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐┌─┴─┐  │  »
«q_3: ┤ P(2*(π - x[0])*(π - x[3])) ├─────┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘     └───┘└───┘┌─┴─┐»
«q_4: ─────────────────────────────────────────────┤ X ├»
«                                                  └───┘»
«q_5: ──────────────────────────────────────────────────»
«                                                       »
«q_6: ──────────────────────────────────────────────────»
«                                                       »
«q_7: ──────────────────────────────────────────────────»
«                                                       »
«                                                            »
«q_0: ─────────────────────────────────────■──────────────■──»
«                                          │              │  »
«q_1: ────────────────────────────────■────┼─────────■────┼──»
«                                     │    │         │    │  »
«q_2: ────────────────────────────────┼────┼────■────┼────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │  ┌─┴─┐  │    │  »
«q_3: ┤ P(2*(π - x[1])*(π - x[3])) ├┤ X ├──┼──┤ X ├──┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐└───┘┌─┴─┐  │  »
«q_4: ┤ P(2*(π - x[0])*(π - x[4])) ├─────┤ X ├─────┤ X ├──┼──»
«     └────────────────────────────┘     └───┘     └───┘┌─┴─┐»
«q_5: ──────────────────────────────────────────────────┤ X ├»
«                                                       └───┘»
«q_6: ───────────────────────────────────────────────────────»
«                                             

In [ ]:
bundle = torchaudio.pipelines.WAV2VEC2_BASE
wav2vec = bundle.get_model().to(device_cnn)
wav2vec.eval()

for p in wav2vec.parameters():
    p.requires_grad = False

In [ ]:
class HybridWav2VecHQCNN(nn.Module):
    def __init__(self, wav2vec, qnn, feature_dim, num_classes):
        super().__init__()
        self.wav2vec = wav2vec

        self.reduce = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Linear(128, feature_dim),
            nn.Tanh()
        )

        self.qnn = TorchConnector(qnn)
        self.fc = nn.Linear(1, num_classes)

    def forward(self, wav):
        wav = wav.to(device_cnn)

        with torch.no_grad():
            features, _ = self.wav2vec(wav)

        emb = features.mean(dim=1)
        x = self.reduce(emb)

        x = x.to(device_qnn)
        x = self.qnn(x)
        return self.fc(x)

In [ ]:
model = HybridWav2VecHQCNN(
    wav2vec, qnn, FEATURE_DIM, NUM_CLASSES
)

model.reduce.to(device_cnn)
model.qnn.to(device_qnn)
model.fc.to(device_qnn)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=LR)

In [ ]:
# ============================================================
# Audio loader (FIXED LENGTH for WAV2VEC2)
# ============================================================
def load_wav_fixed(path, target_sr=16000, clip_seconds=4.0):
    fixed_len = int(target_sr * clip_seconds)

    wav_np, sr = sf.read(path)
    wav = torch.tensor(wav_np).float()

    # stereo → mono
    if wav.ndim == 2:
        wav = wav.mean(dim=1)

    wav = wav.unsqueeze(0)  # [1, T]

    if sr != target_sr:
        wav = torchaudio.functional.resample(
            wav, orig_freq=sr, new_freq=target_sr
        )

    # normalize
    wav = wav / (wav.abs().max() + 1e-9)

    T = wav.shape[-1]
    if T < fixed_len:
        wav = torch.nn.functional.pad(wav, (0, fixed_len - T))
    else:
        wav = wav[:, :fixed_len]

    return wav.squeeze(0)  # [T]

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for wavs, labels in train_loader:
        labels = labels.to(device_qnn)

        optimizer.zero_grad()
        logits = model(wavs)          # wavs: [B, T]
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()

    acc = 100.0 * correct / max(total, 1)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | Acc: {acc:.2f}%")
    torch.save(model.state_dict(), "heartsound_hybrid_wav2vec_qnn_checkpoint.pt")

Epoch 1/1 | Loss: 82.1315 | Acc: 26.49%


In [ ]:
torch.save(model.state_dict(), "heartsound_hybrid_panns_cnn14_qnn.pt")

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for wavs, labels in val_loader:
        logits = model(wavs)
        preds = logits.argmax(dim=1)
        y_true.extend(labels)
        y_pred.extend(preds.cpu().numpy())

labels_list = list(range(NUM_CLASSES))

print(classification_report(
    y_true,
    y_pred,
    labels=labels_list,
    target_names=full_dataset.class_names,
    zero_division=0
))

print(confusion_matrix(
    y_true,
    y_pred,
    labels=labels_list
))

                               precision    recall  f1-score   support

                       Asthma       0.37      1.00      0.54        25
     Asthma and Lung Fibrosis       0.00      0.00      0.00         0
                   Bronchitis       0.00      0.00      0.00         3
                         COPD       0.00      0.00      0.00         1
                Heart Failure       0.00      0.00      0.00        11
         Heart Failure + COPD       0.00      0.00      0.00         2
Heart Failure + Lung Fibrosis       0.00      0.00      0.00         0
                Lung Fibrosis       0.00      0.00      0.00         3
                       Normal       0.00      0.00      0.00        20
                    Pneumonia       0.00      0.00      0.00         3

                     accuracy                           0.37        68
                    macro avg       0.04      0.10      0.05        68
                 weighted avg       0.14      0.37      0.20        68

[[2